In [1]:
! pip install spacy rdflib pyshacl requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\sayil\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 3.9 MB/s eta 0:00:04
     ------------------ --------------------- 5.8/12.8 MB 19.0 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 24.3 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 21.8 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import csv
import re
from pathlib import Path

import spacy

nlp = spacy.load("en_core_web_sm")

INPUT_DIR = Path("data/raw")
OUTPUT_FILE = Path("out/triples.csv")

PREDICATES = {
    "bornIn", "educatedAt", "receivedAward", "worksAt", "doctoralAdvisor",
    "bornOn", "diedOn"
}

ALIASES = {
    "Dr. Khokhar": "Rukhsana Khokhar",
    "Khokhar": "Rukhsana Khokhar",
    "Albert A. Michelson": "Albert Abraham Michelson",
    "Abraham Michelson": "Albert Abraham Michelson",
    "Michelson": "Albert Abraham Michelson",
    "Arthur Compton": "Arthur Holly Compton",
    "Compton": "Arthur Holly Compton",
    "Eugene Wigner": "Eugene Paul Wigner",
    "Paul Wigner": "Eugene Paul Wigner",
    "Wigner": "Eugene Paul Wigner",
    "Hans Bethe": "Hans Albrecht Eduard Bethe",
    "Hans Albrecht": "Hans Albrecht Eduard Bethe",
    "Eduard Bethe": "Hans Albrecht Eduard Bethe",
    "Bethe": "Hans Albrecht Eduard Bethe",
    "Bardeen": "John Bardeen",
}

AWARD_WORDS = {"prize", "award", "medal", "fellowship", "honor", "honour"}
DEGREE_WORDS = {"bachelor", "master", "ph.d", "phd", "doctorate", "degree"}

URL_RE = re.compile(r"\s*\(https?://[^()]*(?:\([^()]*\)[^()]*)*\)")
FOOTNOTE_RE = re.compile(r"\[\d+\]?")
YEAR_RE = re.compile(r"^\d{4}$")
LEADING_ARTICLE_RE = re.compile(r"^(the|a|an)\s+", re.IGNORECASE)
TRAILING_PUNCT_RE = re.compile(r"[\s:;,.]+$")
POSSESSIVE_RE = re.compile(r"['’]s$")
TRAILING_ADVERB_RE = re.compile(r"\s+(twice|once|again|also|too)\s*$", re.IGNORECASE)

MONTH = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
DATE = rf"{MONTH}\s+\d{{1,2}},\s+\d{{4}}"
DATE_RANGE_RE = re.compile(rf"({DATE})\s*[–-]\s*({DATE})", re.IGNORECASE)

FEMALE_NAMES = {"Rukhsana", "she", "her", "hers"}
MALE_NAMES = {"Daniyal", "he", "him", "his"}


class ExtractionState:
    def __init__(self):
        self.merge_count = 0
        self.last_person = None
        self.male_person = None
        self.female_person = None


def read_text_safely(path: Path) -> str:
    raw = path.read_bytes()
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")
    if any(ch in text for ch in "\u00c3\u00c2\u00d4"):
        try:
            text = text.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    text = text.lstrip("\ufeff")
    text = URL_RE.sub("", text)
    return text


def clean(text: str) -> str:
    if not text:
        return text
    text = URL_RE.sub("", text)
    text = FOOTNOTE_RE.sub("", text)
    text = text.replace("]", "")
    text = POSSESSIVE_RE.sub("", text)
    text = LEADING_ARTICLE_RE.sub("", text)
    text = " ".join(text.split())
    text = TRAILING_ADVERB_RE.sub("", text)
    text = TRAILING_PUNCT_RE.sub("", text)
    return text


def normalize(name: str, state: ExtractionState) -> str:
    if not name:
        return name
    name = clean(name)
    for alias, original in ALIASES.items():
        if name.casefold() == alias.casefold():
            if name != original:
                state.merge_count += 1
            return original
    return name


def is_negated(token) -> bool:
    for t in [token] + list(token.children):
        if t.dep_ == "neg" or t.lower_ in {"never", "not", "n't", "no"}:
            return True
    return False


def full_np(token) -> str:
    for ent in token.doc.ents:
        if ent.start <= token.i < ent.end:
            return clean(ent.text)
    for chunk in token.doc.noun_chunks:
        if chunk.start <= token.i < chunk.end:
            return clean(chunk.text)
    return clean(token.text)


def place_np(token) -> str:
    for ent in token.doc.ents:
        if ent.start <= token.i < ent.end and ent.label_ in {"GPE", "LOC", "FAC"}:
            return clean(ent.text)
    text = full_np(token)
    text = re.sub(r"\s+(?:in|at|on)\s+\d{4}.*$", "", text)
    return text


def is_award(text: str) -> bool:
    t = text.casefold()
    return any(w in t for w in AWARD_WORDS) and not any(w in t for w in DEGREE_WORDS)


def is_degree(text: str) -> bool:
    t = text.casefold()
    return any(w in t for w in DEGREE_WORDS)


def update_state_with_entities(doc, state: ExtractionState):
    for ent in doc.ents:
        if ent.label_ == "PERSON" and not is_award(ent.text):
            name = normalize(ent.text, state)
            state.last_person = name
            if any(n in name for n in FEMALE_NAMES):
                state.female_person = name
            elif any(n in name for n in MALE_NAMES):
                state.male_person = name
            else:
                if "Rukhsana" in name:
                    state.female_person = name
                else:
                    state.male_person = name


def resolve_pronoun_subject(doc, state: ExtractionState):
    for token in doc:
        if token.lower_ in {"he", "his", "him"}:
            return state.male_person or state.last_person
        if token.lower_ in {"she", "her", "hers"}:
            return state.female_person or state.last_person
    return None


def extract_dates(doc, subject: str):
    triples = []
    text = doc.text
    m = DATE_RANGE_RE.search(text)
    if m:
        triples.append((subject, "bornOn", clean(m.group(1))))
        triples.append((subject, "diedOn", clean(m.group(2))))
    return triples


def extract_doctoral_advisor(doc, state: ExtractionState):
    triples = []
    for token in doc:
        if token.lemma_ == "be":
            nsubj = [w for w in token.children if w.dep_ == "nsubj"]
            attr = [w for w in token.children if w.dep_ in ("attr", "acomp")]
            if nsubj and attr:
                subj_np = nsubj[0]
                if "advisor" in subj_np.text.lower() or "adviser" in subj_np.text.lower():
                    person = None
                    for child in subj_np.children:
                        if child.dep_ == "poss":
                            person = full_np(child)
                    if person:
                        advisor = full_np(attr[0])
                        if advisor:
                            triples.append((
                                normalize(person, state), 
                                "doctoralAdvisor", 
                                normalize(advisor, state)
                            ))
    return triples


def deduplicate_triples(triples):
    triples = list(dict.fromkeys(triples))
    grouped = {}
    for s, p, o in triples:
        grouped.setdefault((s, p), []).append(o)
        
    final_triples = []
    for (s, p), objs in grouped.items():
        objs = sorted(list(set(objs)), key=len, reverse=True)
        kept = []
        for obj in objs:
            if any(obj.lower() in k.lower() for k in kept):
                continue
            kept.append(obj)
        for k in kept:
            final_triples.append((s, p, k))
    return final_triples


def extract_sentence(doc, state: ExtractionState):
    triples = []
    update_state_with_entities(doc, state)
    
    subject = resolve_pronoun_subject(doc, state)
    
    if not subject:
        persons = [e for e in doc.ents if e.label_ == "PERSON" and not is_award(e.text)]
        if persons:
            subject = normalize(persons[0].text, state)
        elif state.last_person:
            subject = state.last_person

    if not subject:
        return triples

    subject = normalize(subject, state)
    
    triples.extend(extract_dates(doc, subject))
    triples.extend(extract_doctoral_advisor(doc, state))

    for token in doc:
        if is_negated(token):
            continue
        lemma = token.lemma_.lower()

        if lemma == "bear":
            for child in token.children:
                if child.dep_ in {"prep", "agent"} and child.lower_ in {"in", "at"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            place = place_np(obj)
                            if place and not YEAR_RE.match(place):
                                triples.append((subject, "bornIn", normalize(place, state)))

        if lemma in {"study", "graduate", "attend"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "in", "from"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            triples.append((subject, "educatedAt", normalize(full_np(obj), state)))

        if lemma == "earn" and any(
            child.dep_ in {"dobj", "obj"} and is_degree(full_np(child))
            for child in token.children
        ):
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ == "from":
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            triples.append((subject, "educatedAt", normalize(full_np(obj), state)))

        if lemma in {"receive", "win", "earn", "award"}:
            for child in token.children:
                if child.dep_ in {"dobj", "obj", "attr"}:
                    obj_text = full_np(child)
                    if is_award(obj_text):
                        triples.append((subject, "receivedAward", normalize(obj_text, state)))

        if lemma in {"join", "work", "employ"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "for"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            obj_text = full_np(obj)
                            if obj_text and not YEAR_RE.match(obj_text):
                                triples.append((subject, "worksAt", normalize(obj_text, state)))
                if lemma == "join" and child.dep_ in {"dobj", "obj"}:
                    obj_text = full_np(child)
                    if obj_text and not YEAR_RE.match(obj_text):
                        triples.append((subject, "worksAt", normalize(obj_text, state)))
                if child.dep_ == "prep" and child.lower_ == "as":
                    for role in child.children:
                        for sub in role.children:
                            if sub.dep_ == "prep" and sub.lower_ == "at":
                                for obj in sub.children:
                                    if obj.dep_ == "pobj":
                                        triples.append((subject, "worksAt", normalize(full_np(obj), state)))

        if lemma in {"professor", "researcher", "lecturer"}:
            for child in token.children:
                if child.dep_ == "prep" and child.lower_ in {"at", "in"}:
                    for obj in child.children:
                        if obj.dep_ == "pobj":
                            triples.append((subject, "worksAt", normalize(full_np(obj), state)))

    return deduplicate_triples(triples)


def main():
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    all_rows = []
    state = ExtractionState()

    for file_path in sorted(INPUT_DIR.glob("*.txt")):
        text = read_text_safely(file_path)
        doc = nlp(text)

        for sent in doc.sents:
            triples = extract_sentence(sent.as_doc(), state)
            for subject, predicate, obj in triples:
                if predicate in PREDICATES and subject and obj:
                    all_rows.append({
                        "subject": subject,
                        "predicate": predicate,
                        "object": obj,
                        "source_doc": file_path.name,
                        "sentence": clean(sent.text),
                    })

    columns = ["subject", "predicate", "object", "source_doc", "sentence"]
    with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(all_rows)

    print("Extraction completed!")
    print("Total triples:", len(all_rows))
    print("Alias merges reported:", state.merge_count)
    print("Output:", OUTPUT_FILE)


if __name__ == "__main__":
    main()

Extraction completed!
Total triples: 23
Alias merges reported: 9
Output: out\triples.csv
